#### 06 — Fine-tuning Transformer for Hallucination Detection

In this notebook we fine-tune a transformer encoder on the hallucination
detection task using prompt–response pairs.

Goal:
Evaluate whether task-adapted representation learning can surpass
strong classical baselines (TF-IDF + numeric features).

In [ ]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

from src.data.load_splits import load_splits


In [ ]:
train_df, val_df, test_df = load_splits(ROOT)

print(train_df.shape, val_df.shape, test_df.shape)
train_df[["prompt", "response", "label"]].head(2)


#### Build model input

In [ ]:
SEP_TOKEN = " [SEP] "

def build_input_text(df):
    return (df["prompt"].fillna("") + SEP_TOKEN + df["response"].fillna("")).tolist()

train_texts = build_input_text(train_df)
val_texts   = build_input_text(val_df)
test_texts  = build_input_text(test_df)

train_labels = train_df["label"].values
val_labels   = val_df["label"].values
test_labels  = test_df["label"].values


#### Dataset class

In [ ]:
class HalluDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.enc = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=max_length,
        )
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


#### Tokenizer & datasets

In [ ]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

MAX_LEN = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_ds = HalluDataset(train_texts, train_labels, tokenizer, max_length=MAX_LEN)
val_ds   = HalluDataset(val_texts,   val_labels,   tokenizer, max_length=MAX_LEN)
test_ds  = HalluDataset(test_texts,  test_labels,  tokenizer, max_length=MAX_LEN)


#### Metrics

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary", zero_division=0
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall,
    }


#### MODEL

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
)


#### Training arguments

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="../models/finetuned_transformer",
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,

    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    logging_steps=100,
    report_to="none",

    remove_unused_columns=False,
    dataloader_num_workers=0,
    dataloader_pin_memory=False,   # silence MPS warning

    fp16=False,                    # keep false on Mac/MPS unless you're sure
)



#### Trainer

In [ ]:
from transformers import Trainer, EarlyStoppingCallback

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)


In [ ]:
import torch

use_mps = torch.backends.mps.is_available()
device = "mps" if use_mps else "cpu"
print("Using device:", device)


#### Train

In [ ]:
trainer.train()

#### EVAL

In [ ]:
val_metrics  = trainer.evaluate(val_ds)
test_metrics = trainer.evaluate(test_ds)

print("Validation:", val_metrics)
print("Test:", test_metrics)

trainer.save_model("../models/finetuned_transformer/best_model")